# Multi-Agent B2B Customer Support & Performance Analytics Platform
## Progetto Finale - AI Laboratory

Questo notebook implementa un'architettura **Multi-Agent** con pattern **Human-in-the-Loop (HITL)** per la gestione automatizzata dell'assistenza clienti e dell'analisi delle performance in un'agenzia di servizi digitali e crescita social.

### Principali Passaggi Architetturali:
1. **Context Isolation**: Separazione dei tool e dei domini di conoscenza per ciascun agente specializzato.
2. **Standardized Communication Protocol**: Schema dati rigido (`Finding`, `TraceEvent`, `EscalationReport`) per un dialogo deterministico.
3. **Specialized Agent Loops**: Agenti dedicati per Growth Analytics, Project Status/FAQ ed Escalation.
4. **Manager / Orchestrator**: Triage e instradamento dinamico con esecuzione concorrente e sintesi dei risultati.
5. **Observability & Traceability**: Audit Trail completo di tutte le chiamate ai tool e decisioni, con escalation automatica per casi ad alto impatto o sentiment negativo.

In [1]:
import asyncio
import json
from datetime import datetime
from enum import Enum
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

print("Librerie importate con successo.")

Librerie importate con successo.


### Step 1: Standardized Communication Protocol (Contratto Dati Comune)
Definizione delle strutture dati immutabili per lo scambio informativo inter-agente e per l'Audit Log.

In [2]:
class AgentStatus(str, Enum):
    NORMAL = "normal"
    WARNING = "warning"
    CRITICAL = "critical"

class Finding(BaseModel):
    agent: str = Field(description="Nome dell'agente specializzato")
    status: AgentStatus = Field(description="Stato diagnostico rilevato")
    component: str = Field(description="Sotto-dominio o servizio analizzato")
    finding: str = Field(description="Sintesi della diagnosi o risposta")
    evidence: Dict[str, Any] = Field(default_factory=dict, description="Dati oggettivi e metriche a supporto")
    confidence: float = Field(ge=0.0, le=1.0, description="Livello di certezza del risultato (0.0-1.0)")

class TraceEvent(BaseModel):
    timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())
    agent: str
    action: str
    details: Dict[str, Any]

class EscalationReport(BaseModel):
    client_id: str
    reason: str
    sentiment_score: float
    findings_summary: List[Finding]
    recommended_action: str
    requires_human_approval: bool = True

### Step 2: Context Isolation & Mock Tools
Isolamento dei tool per ciascun sotto-dominio al fine di evitare allucinazioni ed eliminare incroci indebiti di dati.

In [3]:
class GrowthAnalyticsTools:
    """Tool isolati per l'analisi della crescita e delle metriche social."""
    @staticmethod
    def get_account_growth_stats(client_id: str) -> Dict[str, Any]:
        mock_database = {
            "client_001": {
                "followers_total": 14500,
                "followers_gained_30d": 1200,
                "growth_rate": "+9.0%",
                "engagement_rate": "4.8%",
                "reach_status": "healthy"
            },
            "client_002": {
                "followers_total": 8200,
                "followers_gained_30d": -50,
                "growth_rate": "-0.6%",
                "engagement_rate": "1.2%",
                "reach_status": "drop_detected"
            }
        }
        return mock_database.get(client_id, {"error": "Client ID non trovato"})

class ProjectFAQTools:
    """Tool isolati per la consultazione dello stato progetti e Knowledge Base."""
    @staticmethod
    def get_project_status(client_id: str) -> Dict[str, Any]:
        mock_projects = {
            "client_001": {
                "project_type": "Mobile App iOS/Android",
                "current_phase": "Testing & QA",
                "completion_percentage": 85,
                "estimated_delivery": "2026-10-25"
            },
            "client_002": {
                "project_type": "E-Commerce Web Portal",
                "current_phase": "UX/UI Design Approval",
                "completion_percentage": 40,
                "estimated_delivery": "2026-11-15"
            }
        }
        return mock_projects.get(client_id, {"error": "Progetto non trovato"})

    @staticmethod
    def search_knowledge_base(query: str) -> str:
        kb = {
            "tempi": "I progetti software standard hanno tempi di sviluppo dai 2 ai 4 mesi in base alla complessità.",
            "modalità": "Lavoriamo con metodologia Agile / Scrum con sprint di due settimane e report periodici.",
            "growth": "La strategia di crescita social si basa su ottimizzazione del profilo, hashtag audit e interazioni mirate B2B."
        }
        for k, v in kb.items():
            if k in query.lower():
                return v
        return "Informazione non presente nella KB standard. Consultare il referente aziendale."

class EscalationTools:
    """Tool per l'invio di notifiche e briefing al team umano (HITL)."""
    @staticmethod
    def trigger_human_escalation(report: EscalationReport) -> Dict[str, Any]:
        return {
            "ticket_id": f"TICKET-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
            "status": "ESCALATED_TO_HUMAN",
            "assigned_to": "Account Manager / Team Lead",
            "briefing": report.model_dump()
        }

### Step 3: Agent Loops (Agenti Specialistici)
Ogni agente esegue la logica diagnostica di dominio e restituisce un `Finding`.

In [4]:
class GrowthAnalyticsAgent:
    def __init__(self, trace_log: List[TraceEvent]):
        self.name = "Growth & Analytics Specialist"
        self.trace_log = trace_log

    async def analyze(self, client_id: str, query: str) -> Finding:
        self.trace_log.append(TraceEvent(agent=self.name, action="fetch_growth_stats", details={"client_id": client_id}))
        stats = GrowthAnalyticsTools.get_account_growth_stats(client_id)

        if "error" in stats:
            return Finding(agent=self.name, status=AgentStatus.CRITICAL, component="Social Growth Analytics", finding="Cliente non registrato nel sistema analytics.", evidence=stats, confidence=0.95)

        is_drop = stats.get("reach_status") == "drop_detected"
        status = AgentStatus.WARNING if is_drop else AgentStatus.NORMAL
        finding_text = f"Rilevato calo nelle performance social (Crescita: {stats['growth_rate']}, Engagement: {stats['engagement_rate']})." if is_drop else f"Crescita regolare. Guadagnati {stats['followers_gained_30d']} follower negli ultimi 30 giorni (Rateo: {stats['growth_rate']})."

        return Finding(agent=self.name, status=status, component="Social Growth Analytics", finding=finding_text, evidence=stats, confidence=0.92)

class ProjectFAQAgent:
    def __init__(self, trace_log: List[TraceEvent]):
        self.name = "Project & FAQ Specialist"
        self.trace_log = trace_log

    async def analyze(self, client_id: str, query: str) -> Finding:
        self.trace_log.append(TraceEvent(agent=self.name, action="check_project_status_and_kb", details={"client_id": client_id, "query": query}))
        project_data = ProjectFAQTools.get_project_status(client_id)
        kb_answer = ProjectFAQTools.search_knowledge_base(query)

        finding_text = f"Progetto '{project_data.get('project_type', 'N/D')}': Stato '{project_data.get('current_phase', 'N/D')}' ({project_data.get('completion_percentage', 0)}% completato). Delivery stimata: {project_data.get('estimated_delivery', 'N/D')}."
        if "Informazione non presente" not in kb_answer:
            finding_text += f" Info KB: {kb_answer}"

        return Finding(agent=self.name, status=AgentStatus.NORMAL, component="Software Project Management", finding=finding_text, evidence={"project_data": project_data, "kb_answer": kb_answer}, confidence=0.90)

### Step 4: Manager / Orchestrator con Human-in-the-Loop
Gestione del triage, della concorrenza via `asyncio`, della valutazione delle soglie di rischio e dell'escalation.

In [5]:
class TriageOrchestrator:
    def __init__(self):
        self.trace_log: List[TraceEvent] = []
        self.growth_agent = GrowthAnalyticsAgent(self.trace_log)
        self.project_agent = ProjectFAQAgent(self.trace_log)

    def _analyze_sentiment(self, text: str) -> float:
        negative_keywords = ["pessimo", "deluso", "disdetta", "reclamo", "responsabile", "soldi sprecati", "insoddisfatto", "annullare"]
        score = 0.8
        for word in negative_keywords:
            if word in text.lower():
                score -= 0.25
        return max(0.0, min(1.0, score))

    async def process_customer_message(self, client_id: str, message: str) -> Dict[str, Any]:
        self.trace_log.append(TraceEvent(agent="Orchestrator", action="receive_message", details={"client_id": client_id, "message": message}))

        sentiment_score = self._analyze_sentiment(message)
        requires_escalation = sentiment_score < 0.50 or any(kw in message.lower() for kw in ["responsabile", "disdetta", "reclamo"])

        tasks = [self.growth_agent.analyze(client_id, message), self.project_agent.analyze(client_id, message)]
        findings: List[Finding] = await asyncio.gather(*tasks)

        for f in findings:
            if f.status in [AgentStatus.WARNING, AgentStatus.CRITICAL] or f.confidence < 0.75:
                requires_escalation = True

        if requires_escalation:
            report = EscalationReport(
                client_id=client_id,
                reason="Sentiment negativo o anomalia rilevata nelle metriche/progetto.",
                sentiment_score=sentiment_score,
                findings_summary=findings,
                recommended_action="Contattare direttamente il cliente via messaggistica/telefono con briefing pronto.",
                requires_human_approval=True
            )
            escalation_result = EscalationTools.trigger_human_escalation(report)
            self.trace_log.append(TraceEvent(agent="Orchestrator", action="trigger_escalation", details=escalation_result))

            return {
                "status": "HUMAN_ESCALATION_TRIGGERED",
                "customer_response": "Ho preso in carico la tua richiesta e ho inoltrato la scheda dettagliata del tuo profilo al nostro team. Un nostro responsabile ti risponderà qui a breve!",
                "internal_escalation_briefing": escalation_result
            }
        else:
            summary_response = "Gentile cliente,\nEcco l'aggiornamento richiesto:\n"
            for f in findings:
                summary_response += f"• **{f.component}**: {f.finding}\n"
            return {
                "status": "AUTOMATED_RESPONSE",
                "customer_response": summary_response,
                "findings": [f.model_dump() for f in findings]
            }

### Step 5: Demo & Test Cases
Esecuzione degli scenari di test per validare sia l'automazione che l'escalation Human-in-the-Loop con l'Audit Log.

In [6]:
async def run_demo():
    orchestrator = TriageOrchestrator()

    print("="*70)
    print("SCENARIO 1: Richiesta Informazioni Ordinaria (Client 001)")
    print("="*70)
    res1 = await orchestrator.process_customer_message("client_001", "Ciao! Volevo sapere a che punto è lo sviluppo dell'app e come sta andando la crescita social.")
    print(f"Stato: {res1['status']}")
    print(f"Risposta al Cliente:\n{res1['customer_response']}\n")

    print("="*70)
    print("SCENARIO 2: Reclamo e Frustrazione - Attivazione Escalation Human-in-the-Loop (Client 002)")
    print("="*70)
    res2 = await orchestrator.process_customer_message("client_002", "Sono molto deluso e insoddisfatto dei risultati della crescita! I follower stanno calando, voglio parlare subito con un responsabile per disdetta!")
    print(f"Stato: {res2['status']}")
    print(f"Risposta al Cliente:\n{res2['customer_response']}\n")
    print("--- BRIEFING INTERNO GENERATO PER L'OPERATORE UMANO ---")
    print(json.dumps(res2['internal_escalation_briefing'], indent=2, ensure_ascii=False))
    print("\n" + "="*70)
    print("AUDIT TRAIL (TRACCIA DELLE AZIONI DEGLI AGENTI)")
    print("="*70)
    for event in orchestrator.trace_log:
        print(f"[{event.timestamp}] [{event.agent}] -> {event.action}: {event.details}")

await run_demo()

SCENARIO 1: Richiesta Informazioni Ordinaria (Client 001)
Stato: AUTOMATED_RESPONSE
Risposta al Cliente:
Gentile cliente,
Ecco l'aggiornamento richiesto:
• **Social Growth Analytics**: Crescita regolare. Guadagnati 1200 follower negli ultimi 30 giorni (Rateo: +9.0%).
• **Software Project Management**: Progetto 'Mobile App iOS/Android': Stato 'Testing & QA' (85% completato). Delivery stimata: 2026-10-25.


SCENARIO 2: Reclamo e Frustrazione - Attivazione Escalation Human-in-the-Loop (Client 002)
Stato: HUMAN_ESCALATION_TRIGGERED
Risposta al Cliente:
Ho preso in carico la tua richiesta e ho inoltrato la scheda dettagliata del tuo profilo al nostro team. Un nostro responsabile ti risponderà qui a breve!

--- BRIEFING INTERNO GENERATO PER L'OPERATORE UMANO ---
{
  "ticket_id": "TICKET-20261002-192532",
  "status": "ESCALATED_TO_HUMAN",
  "assigned_to": "Account Manager / Team Lead",
  "briefing": {
    "client_id": "client_002",
    "reason": "Sentiment negativo o anomalia rilevata nelle m